# Occupancy Transformer Training (Google Colab)

This notebook trains a PyTorch Transformer model for next-hour occupancy prediction.

Input files in Google Drive:

- `PIRSensorData.csv`
- `hotelReservationData.csv`

Outputs:

- `occupancy_transformer.pt`
- `occupancy_transformer_metadata.json`
- `occupancy_transformer_report.txt`
- `occupancy_transformer_confusion_matrix.csv`
- `occupancy_transformer_outputs.zip`


## Optional GitHub Clone
This notebook is self-contained, but this cell also clones the project code so your Colab runtime follows the same GitHub + Drive workflow.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/VisualizationApp.git'  # <-- change this
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


## 1. Runtime

Set `Runtime` -> `Change runtime type` -> `GPU` before training.

In [1]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-80GB


## 2. Mount Google Drive and Locate Data

If the automatic search finds multiple files, set `KNOWN_PIR_CSV` and `KNOWN_RESERVATION_CSV` manually.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

KNOWN_PIR_CSV = None
KNOWN_RESERVATION_CSV = None
# Example:
KNOWN_PIR_CSV = '/content/drive/MyDrive/VisualizationApp/Data/PIRSensorData.csv'
KNOWN_RESERVATION_CSV = '/content/drive/MyDrive/VisualizationApp/Data/hotelReservationData.csv'

def find_drive_file(filename):
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} in /content/drive/MyDrive')
    print(f'Found {filename}:')
    for i, match in enumerate(matches):
        print(i, match)
    return str(matches[0])

PIR_CSV = KNOWN_PIR_CSV or find_drive_file('PIRSensorData.csv')
RESERVATION_CSV = KNOWN_RESERVATION_CSV or find_drive_file('hotelReservationData.csv')
OUT_DIR = '/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/OccupancyPrediction/trandformer'

print('PIR_CSV =', PIR_CSV)
print('RESERVATION_CSV =', RESERVATION_CSV)
print('OUT_DIR =', OUT_DIR)


Mounted at /content/drive
PIR_CSV = /content/drive/MyDrive/VisualizationApp/PIRSensorData.csv
RESERVATION_CSV = /content/drive/MyDrive/VisualizationApp/hotelReservationData.csv
OUT_DIR = /content/occupancy_transformer_outputs


## 3. Training Configuration

`MAX_SEQUENCES` limits memory use. Increase it if Colab has enough RAM.

In [3]:
MAX_ROWS = None          # Optional PIR row limit for quick tests.
MAX_SEQUENCES = 400000   # Limit generated training sequences to control RAM.
STRIDE = 3               # Use every Nth timestamp per room. Lower = more data, slower.
SEQ_LEN = 12             # Last 12 five-minute samples = 1 hour history.
HORIZON_STEPS = 12       # Predict 12 five-minute samples ahead = 1 hour.

EPOCHS = 20
BATCH_SIZE = 512
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.15
LR = 3e-4
WEIGHT_DECAY = 1e-2
TRAIN_FRACTION = 0.8
SEED = 42


In [4]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

ROOM_TYPES = ['Deluxe', 'Standard', 'Suite', 'Unknown']

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_merged_pir_data(pir_csv, reservation_csv, max_rows=None):
    pir = pd.read_csv(
        pir_csv,
        usecols=['timestamp', 'room_number', 'pir_motion', 'room_state', 'adults', 'children', 'guest_id'],
        parse_dates=['timestamp'],
        nrows=max_rows,
    )
    pir = pir.sort_values(['room_number', 'timestamp']).reset_index(drop=True)
    pir['guest_id'] = pd.to_numeric(pir['guest_id'], errors='coerce')
    pir['active_guest'] = pir['guest_id'].notna().astype(float)
    pir['pir_motion'] = pd.to_numeric(pir['pir_motion'], errors='coerce').fillna(0).astype(float)
    pir['adults'] = pd.to_numeric(pir['adults'], errors='coerce').fillna(0).astype(float)
    pir['children'] = pd.to_numeric(pir['children'], errors='coerce').fillna(0).astype(float)
    pir['guest_count'] = pir['adults'] + pir['children']

    reservations = pd.read_csv(
        reservation_csv,
        usecols=['Guest ID', 'Nationality', 'Room Type', 'Total Nights', 'Total Amount', 'Stay_Duration'],
    ).rename(columns={'Guest ID': 'guest_id', 'Room Type': 'room_type'})
    reservations['guest_id'] = pd.to_numeric(reservations['guest_id'], errors='coerce')
    reservations['Total Nights'] = pd.to_numeric(reservations['Total Nights'], errors='coerce').fillna(0)
    reservations['Total Amount'] = pd.to_numeric(reservations['Total Amount'], errors='coerce').fillna(0)
    reservations['Stay_Duration'] = pd.to_numeric(reservations['Stay_Duration'], errors='coerce').replace(0, np.nan)
    reservations['price_per_night'] = reservations['Total Amount'] / reservations['Stay_Duration']
    reservations['Stay_Duration'] = reservations['Stay_Duration'].fillna(0)
    reservations['price_per_night'] = reservations['price_per_night'].fillna(0)
    reservations['room_type'] = reservations['room_type'].fillna('Unknown')

    df = pir.merge(reservations, on='guest_id', how='left')
    df['room_type'] = df['room_type'].fillna('Unknown')
    df['Total Nights'] = df['Total Nights'].fillna(0)
    df['Total Amount'] = df['Total Amount'].fillna(0)
    df['Stay_Duration'] = df['Stay_Duration'].fillna(0)
    df['price_per_night'] = df['price_per_night'].fillna(0)
    return df

def add_features(df):
    df = df.copy()
    df['hour'] = df['timestamp'].dt.hour
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['Total Nights'] = df['Total Nights'] / 30.0
    df['Total Amount'] = df['Total Amount'] / 2000.0
    df['Stay_Duration'] = df['Stay_Duration'] / 30.0
    df['price_per_night'] = df['price_per_night'] / 500.0
    for room_type in ROOM_TYPES:
        df[f'room_type_{room_type}'] = (df['room_type'].eq(room_type)).astype(float)
    feature_cols = [
        'pir_motion', 'adults', 'children', 'guest_count', 'active_guest',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
        'Total Nights', 'Total Amount', 'Stay_Duration', 'price_per_night',
    ] + [f'room_type_{room_type}' for room_type in ROOM_TYPES]
    return df, feature_cols

def build_sequences(df, feature_cols, seq_len, horizon_steps, stride=1, max_sequences=None):
    xs, ys = [], []
    for _, room_df in df.groupby('room_number', sort=False):
        room_df = room_df.sort_values('timestamp').reset_index(drop=True)
        features = room_df[feature_cols].to_numpy(dtype=np.float32)
        targets = room_df['room_state'].to_numpy()
        last_start = len(room_df) - horizon_steps
        for idx in range(seq_len - 1, last_start, stride):
            target = targets[idx + horizon_steps]
            if pd.isna(target):
                continue
            xs.append(features[idx - seq_len + 1:idx + 1])
            ys.append(target)
            if max_sequences and len(xs) >= max_sequences:
                return np.stack(xs), np.array(ys)
    return np.stack(xs), np.array(ys)


In [5]:
class OccupancyTransformer(nn.Module):
    def __init__(self, input_dim, n_classes, seq_len=SEQ_LEN, d_model=64, n_heads=4, n_layers=2, dim_feedforward=128, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        hidden = self.input_proj(x) + self.pos_embed
        hidden = self.encoder(hidden)
        hidden = self.norm(hidden)
        pooled = hidden.mean(dim=1)
        return self.head(pooled)


## 4. Build Sequences and Train


In [6]:
set_seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

df = load_merged_pir_data(PIR_CSV, RESERVATION_CSV, max_rows=MAX_ROWS)
df, feature_cols = add_features(df)
x, y_str = build_sequences(
    df,
    feature_cols,
    seq_len=SEQ_LEN,
    horizon_steps=HORIZON_STEPS,
    stride=STRIDE,
    max_sequences=MAX_SEQUENCES,
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_str)
classes = list(label_encoder.classes_)
split_idx = max(1, min(len(x) - 1, int(len(x) * TRAIN_FRACTION)))
x_train, x_test = x[:split_idx], x[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

counts = np.bincount(y_train, minlength=len(classes)).clip(min=1)
weights = torch.tensor(len(y_train) / (len(classes) * counts), dtype=torch.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('device:', device)
print('sequence tensor:', x.shape)
print('train:', len(x_train), 'test:', len(x_test))
print('classes:', classes)
print('feature columns:', feature_cols)

model = OccupancyTransformer(
    input_dim=x.shape[-1],
    n_classes=len(classes),
    seq_len=SEQ_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss(weight=weights.to(device))

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train).long()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
x_test_tensor = torch.from_numpy(x_test).to(device)

best_acc = -1.0
best_state = None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)

    model.eval()
    with torch.no_grad():
        pred = model(x_test_tensor).argmax(dim=1).cpu().numpy()
    acc = float((pred == y_test).mean())
    if acc > best_acc:
        best_acc = acc
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'epoch {epoch:03d} train_loss={total_loss / len(x_train):.4f} test_acc={acc:.3f}')

print('best test accuracy:', best_acc)


device: cuda
sequence tensor: (400000, 12, 17)
train: 320000 test: 80000
classes: [np.str_('Cleaning'), np.str_('Occupied'), np.str_('Vacant')]
feature columns: ['pir_motion', 'adults', 'children', 'guest_count', 'active_guest', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'Total Nights', 'Total Amount', 'Stay_Duration', 'price_per_night', 'room_type_Deluxe', 'room_type_Standard', 'room_type_Suite', 'room_type_Unknown']


/tmp/ipykernel_672/2782191769.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


epoch 001 train_loss=0.5087 test_acc=0.945
epoch 005 train_loss=0.3236 test_acc=0.976
epoch 010 train_loss=0.3178 test_acc=0.976
epoch 015 train_loss=0.2784 test_acc=0.976
epoch 020 train_loss=0.2681 test_acc=0.976
best test accuracy: 0.98845


## 5. Save Model, Results, and Zip to Drive


In [7]:
model.load_state_dict(best_state)
model.to(device)
model.eval()
with torch.no_grad():
    y_pred = model(x_test_tensor).argmax(dim=1).cpu().numpy()

report = classification_report(
    y_test,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=3,
    zero_division=0,
)
confusion = confusion_matrix(y_test, y_pred, labels=range(len(classes)))
cm_df = pd.DataFrame(confusion, index=classes, columns=classes)
cm_df.index.name = 'true'

out_dir = Path(OUT_DIR)
model_path = out_dir / 'occupancy_transformer.pt'
metadata_path = out_dir / 'occupancy_transformer_metadata.json'
report_path = out_dir / 'occupancy_transformer_report.txt'
cm_path = out_dir / 'occupancy_transformer_confusion_matrix.csv'

report_text = '\n'.join([
    f'sequences: {len(x):,}',
    f'train sequences: {len(x_train):,}',
    f'test sequences: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'horizon steps: {HORIZON_STEPS}',
    f'horizon minutes: {HORIZON_STEPS * 5}',
    f'input features: {len(feature_cols)}',
    f'best test accuracy: {best_acc:.3f}',
    '',
    report,
])
report_path.write_text(report_text)
cm_df.to_csv(cm_path)
torch.save(
    {
        'model_state_dict': best_state,
        'classes': classes,
        'feature_columns': feature_cols,
        'seq_len': SEQ_LEN,
        'horizon_steps': HORIZON_STEPS,
        'horizon_minutes': HORIZON_STEPS * 5,
        'input_dim': x.shape[-1],
        'room_types': ROOM_TYPES,
        'config': {
            'd_model': D_MODEL,
            'n_heads': N_HEADS,
            'n_layers': N_LAYERS,
            'dim_feedforward': DIM_FEEDFORWARD,
            'dropout': DROPOUT,
        },
    },
    model_path,
)
metadata_path.write_text(json.dumps({
    'classes': classes,
    'feature_columns': feature_cols,
    'seq_len': SEQ_LEN,
    'horizon_steps': HORIZON_STEPS,
    'horizon_minutes': HORIZON_STEPS * 5,
    'input_dim': x.shape[-1],
    'room_types': ROOM_TYPES,
    'model_file': model_path.name,
    'report_file': report_path.name,
    'confusion_matrix_file': cm_path.name,
}, indent=2))

print(report_text)
print('saved:', model_path)
print('saved:', metadata_path)
print('saved:', report_path)
print('saved:', cm_path)

zip_path = Path('/content/drive/MyDrive/occupancy_transformer_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, cm_path]:
        zf.write(path, arcname=path.name)
print('saved zip to Drive:', zip_path)


sequences: 400,000
train sequences: 320,000
test sequences: 80,000
sequence length: 12
horizon steps: 12
horizon minutes: 60
input features: 17
best test accuracy: 0.988

              precision    recall  f1-score   support

    Cleaning      0.177     0.414     0.248       140
    Occupied      0.991     0.991     0.991     34532
      Vacant      0.992     0.988     0.990     45328

    accuracy                          0.988     80000
   macro avg      0.720     0.798     0.743     80000
weighted avg      0.990     0.988     0.989     80000

saved: /content/occupancy_transformer_outputs/occupancy_transformer.pt
saved: /content/occupancy_transformer_outputs/occupancy_transformer_metadata.json
saved: /content/occupancy_transformer_outputs/occupancy_transformer_report.txt
saved: /content/occupancy_transformer_outputs/occupancy_transformer_confusion_matrix.csv
saved zip to Drive: /content/drive/MyDrive/occupancy_transformer_outputs.zip
